In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [2]:
df = pd.read_csv("../data/processed/manali_places.csv")

print("Dataset loaded successfully! ✅")
print("Shape:", df.shape)


Dataset loaded successfully! ✅
Shape: (20, 9)


In [3]:
clean_df = df.copy()


In [4]:
text_columns = ["place_id", "name", "address", "country", "category"]

for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

clean_df.head()


,place_id,name,address,country,latitude,longitude,rating,reviews,category
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction


In [5]:
numeric_columns = ["latitude", "longitude", "rating", "reviews"]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df.dtypes


place_id      string
name          string
address       string
country       string
latitude     float64
longitude    float64
rating       float64
reviews        int64
category      string
dtype: object

In [6]:
print("Duplicate full rows:", clean_df.duplicated().sum())
print("Duplicate place IDs:", clean_df["place_id"].duplicated().sum())


Duplicate full rows: 0
Duplicate place IDs: 0


In [7]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates(subset="place_id", keep="first")

after = len(clean_df)

print("Rows before:", before)
print("Rows after:", after)
print("Removed:", before - after)


Rows before: 20
Rows after: 20
Removed: 0


In [8]:
invalid_ratings = clean_df[
    (clean_df["rating"] < 0) |
    (clean_df["rating"] > 5)
]

print("Invalid ratings:", len(invalid_ratings))
invalid_ratings[["name", "rating"]]


Invalid ratings: 0


,name,rating


In [9]:
invalid_reviews = clean_df[clean_df["reviews"] < 0]

print("Invalid review counts:", len(invalid_reviews))
invalid_reviews[["name", "reviews"]]


Invalid review counts: 0


,name,reviews


In [10]:
invalid_coordinates = clean_df[
    (clean_df["latitude"] < -90) |
    (clean_df["latitude"] > 90) |
    (clean_df["longitude"] < -180) |
    (clean_df["longitude"] > 180)
]

print("Invalid coordinate rows:", len(invalid_coordinates))
invalid_coordinates[["name", "latitude", "longitude"]]


Invalid coordinate rows: 0


,name,latitude,longitude


In [11]:
clean_df.isnull().sum().sort_values(ascending=False)


place_id     0
name         0
address      0
country      0
latitude     0
longitude    0
rating       0
reviews      0
category     0
dtype: int64

In [12]:
keywords_to_review = [
    "start point",
    "statue",
    "pardesh",
    "india",
    "valley"
]

pattern = "|".join(keywords_to_review)

review_candidates = clean_df[
    clean_df["name"].str.contains(pattern, case=False, na=False)
]

review_candidates[["name", "address", "category", "rating", "reviews"]]


,name,address,category,rating,reviews
8,Lama Dugh Trek Start Point,"65XG+H26, Old Manali, Manali, Himachal Pradesh...",Tourist attraction,4.6,297
9,Atal Bihari statue,"65WQ+6WM, Siyal, Manali, Himachal Pradesh 1751...",Tourist attraction,4.5,74
10,Kharma valley,"7539+W6C, Manu Temple Rd, Old Manali, Manali, ...",Tourist attraction,4.8,143
13,Himachal PARDESH,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",Tourist attraction,3.9,11


In [13]:
clean_df["data_quality_score"] = 0

clean_df.loc[clean_df["name"].notna(), "data_quality_score"] += 1
clean_df.loc[clean_df["rating"].notna(), "data_quality_score"] += 1
clean_df.loc[clean_df["reviews"].notna(), "data_quality_score"] += 1
clean_df.loc[clean_df["latitude"].notna(), "data_quality_score"] += 1
clean_df.loc[clean_df["longitude"].notna(), "data_quality_score"] += 1
clean_df.loc[clean_df["category"].notna(), "data_quality_score"] += 1

clean_df[
    ["name", "rating", "reviews", "category", "data_quality_score"]
].head(10)


,name,rating,reviews,category,data_quality_score
0,Hadimba Devi Temple,4.6,49688,Tourist attraction,6
1,Old Manali snow point,4.6,428,Tourist attraction,6
2,Nehru Kund,4.4,7767,Tourist attraction,6
3,Kullu Manali River rafting,4.5,88,Tourist attraction,6
4,Jogini Falls,4.6,10842,Tourist attraction,6
5,Van Vihar National Park,4.2,9050,Tourist attraction,6
6,Manali View Point,4.6,87,Tourist attraction,6
7,Rahala Waterfalls,4.5,797,Tourist attraction,6
8,Lama Dugh Trek Start Point,4.6,297,Tourist attraction,6
9,Atal Bihari statue,4.5,74,Tourist attraction,6


In [14]:
clean_df = clean_df.reset_index(drop=True)
clean_df.head()


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6


In [15]:
print("Final shape:", clean_df.shape)

print("\nMissing values:")
print(clean_df.isnull().sum())

print("\nDuplicate place IDs:", clean_df["place_id"].duplicated().sum())

print("\nRating range:", clean_df["rating"].min(), "to", clean_df["rating"].max())
print("Review count range:", clean_df["reviews"].min(), "to", clean_df["reviews"].max())

print(
    "\nCoordinate range:",
    "Latitude", clean_df["latitude"].min(), "to", clean_df["latitude"].max(),
    "| Longitude", clean_df["longitude"].min(), "to", clean_df["longitude"].max()
)


Final shape: (20, 10)

Missing values:
place_id              0
name                  0
address               0
country               0
latitude              0
longitude             0
rating                0
reviews               0
category              0
data_quality_score    0
dtype: int64

Duplicate place IDs: 0

Rating range: 3.9 to 4.8
Review count range: 11 to 49688

Coordinate range: Latitude 32.206785499999995 to 32.3363624 | Longitude 77.1680161 to 77.2251751


In [16]:
output_path = "../data/processed/manali_places_clean.csv"

clean_df.to_csv(output_path, index=False)

print(f"✅ Clean dataset saved to: {output_path}")


✅ Clean dataset saved to: ../data/processed/manali_places_clean.csv
